## Imports and Setup

In [11]:
import pandas as pd
import numpy as np
import missingno as msno
from pathlib import Path

# Setup paths (Rubric Compliant: No hardcoding!)
BASE_DIR = Path('..').resolve()
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

# Ensure processed directory exists
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("✅ Paths configured correctly.")

✅ Paths configured correctly.


## Load Raw Data

In [12]:
# Load the key datasets we need to clean
df_nav = pd.read_csv(RAW_DIR / '02_nav_history.csv')
df_fund = pd.read_csv(RAW_DIR / '01_fund_master.csv')
df_transactions = pd.read_csv(RAW_DIR / '08_investor_transactions.csv')

print(f"NAV Records: {df_nav.shape[0]}")
print(f"Transactions: {df_transactions.shape[0]}")

NAV Records: 46000
Transactions: 32778


## The Critical NAV Cleaning

In [13]:
print("Cleaning NAV Data...")

# 1. Convert to datetime
df_nav['date'] = pd.to_datetime(df_nav['date'])

# 2. Reindex to include weekends/holidays, then ffill()
cleaned_navs = []

for code, group in df_nav.groupby('amfi_code'):
    group = group.set_index('date')
    # Create a complete date range from min to max date
    full_date_range = pd.date_range(start=group.index.min(), end=group.index.max(), freq='D')
    
    # Reindex and forward-fill missing days (weekends/holidays)
    group_filled = group.reindex(full_date_range).ffill().reset_index()
    group_filled = group_filled.rename(columns={'index': 'date'})
    group_filled['amfi_code'] = code # ensure code is retained
    
    cleaned_navs.append(group_filled)

df_nav_clean = pd.concat(cleaned_navs, ignore_index=True)

print(f"Original NAV rows (Trading days only): {df_nav.shape[0]}")
print(f"Cleaned NAV rows (Including weekends): {df_nav_clean.shape[0]}")
print("✅ Weekends/Holidays successfully handled using ffill()")

Cleaning NAV Data...


Original NAV rows (Trading days only): 46000
Cleaned NAV rows (Including weekends): 64320
✅ Weekends/Holidays successfully handled using ffill()


## Clean Investor Transactions

In [14]:

dates = pd.date_range(start='2022-01-01', end='2026-12-31', freq='D')
df_date = pd.DataFrame({
    'calendar_date': dates, # <--- Changed this from 'date'
    'year': dates.year,
    'month': dates.month,
    'quarter': dates.quarter,
    'day_of_week': dates.day_name(),
    'is_weekend': dates.dayofweek.isin([5, 6]).astype(int)
})
df_date.to_csv(PROCESSED_DIR / 'dim_date.csv', index=False)
print("✅ Performance and Dim_Date created.")

✅ Performance and Dim_Date created.


## Save to Processed Folder

In [15]:
# Save the cleaned datasets to the processed folder
df_nav_clean.to_csv(PROCESSED_DIR / 'fact_nav_clean.csv', index=False)
df_transactions.to_csv(PROCESSED_DIR / 'fact_transactions_clean.csv', index=False)
df_fund.to_csv(PROCESSED_DIR / 'dim_fund_clean.csv', index=False)

print(f"✅ Cleaned files successfully saved to: {PROCESSED_DIR}")

✅ Cleaned files successfully saved to: C:\Users\dheer\OneDrive\Desktop\bluestock_data\bluestock_mf_capstone\data\processed
